In [1]:
import os
import random

import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Quantization


In [2]:
def quantize(input_tensor, quantize_to_bits=8):
    @tf.custom_gradient
    def straight_through_estimator(input_tensor):
        max_int = (2 ** (quantize_to_bits - 1)) - 1
        max_float = tf.cast(max_int, tf.float32)
        max_val = tf.reduce_max(tf.abs(input_tensor))
        max_val = tf.maximum(max_val, 1e-8)

        scale = max_float / max_val

        quantized = tf.round(input_tensor * scale)
        quantized = tf.clip_by_value(quantized, -max_float, max_float)
        output = quantized / scale

        def grad(upstream, variables=None):
            if variables is not None:
                return upstream, [None] * len(variables)
            return upstream

        return output, grad

    return straight_through_estimator(input_tensor)

In [3]:
from keras.losses import mean_squared_error


def test_quantization():
    weights = tf.Variable([[1.02583, -0.238905], [-0.05612, -1.2983]], dtype=tf.float32)
    print("Original weights:\n", weights.numpy())

    target = tf.constant([[1.0, 1.0], [1.0, -1.0]], dtype=tf.float32)

    with tf.GradientTape() as tape:
        quantized_weights = quantize(weights, quantize_to_bits=4)
        loss = mean_squared_error(target, quantized_weights)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, weights)

    print("\nQuantized weights:\n", quantized_weights.numpy())
    print("\nGradients:\n", gradients.numpy())


test_quantization()

Original weights:
 [[ 1.02583  -0.238905]
 [-0.05612  -1.2983  ]]

Quantized weights:
 [[ 1.1128286  -0.18547143]
 [-0.         -1.2983    ]]

Gradients:
 [[ 0.05641431 -0.5927357 ]
 [-0.5        -0.14915001]]


# Model Constructing

## Custom Quantized Dense Layer


In [4]:
from keras.initializers import GlorotUniform
from keras.layers import Layer


class QuantizedDense(Layer):
    def __init__(self, units, quantize_to_bits=8, **kwargs):
        super(QuantizedDense, self).__init__(**kwargs)
        self.units = units
        self.quantize_to_bits = quantize_to_bits

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[-1], self.units),
            initializer=GlorotUniform(seed=SEED),
            trainable=True,
        )

        self.bias = self.add_weight(
            name="bias", shape=(self.units,), initializer="zeros", trainable=True
        )

    def call(self, inputs):
        quantized_kernel = quantize(self.kernel, quantize_to_bits=self.quantize_to_bits)
        output = tf.matmul(inputs, quantized_kernel)
        output += self.bias
        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    def get_config(self):
        config = super(QuantizedDense, self).get_config()
        config.update({"units": self.units, "quantize_to_bits": self.quantize_to_bits})
        return config

In [5]:
from keras import Input, Sequential

tf.random.set_seed(42)


def test_quantized_dense():
    model = Sequential([Input(shape=(4,)), QuantizedDense(units=3, quantize_to_bits=4)])

    test_input = tf.random.normal((2, 4))
    test_target = tf.random.normal((2, 3))

    with tf.GradientTape() as tape:
        predictions = model(test_input)
        loss = mean_squared_error(test_target, predictions)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, model.trainable_variables)

    print("\nTest X-Y:\n", f"X: {test_input.numpy()}\n", f"Y: {test_target.numpy()}")

    print("\nModel shape:\n", predictions.shape)
    print("\nWeights:\n", model.layers[0].kernel.numpy())
    print("\nGradients:\n", gradients[0].numpy())


test_quantized_dense()


Test X-Y:
 X: [[ 0.3274685 -0.8426258  0.3194337 -1.4075519]
 [-2.3880599 -1.0392479 -0.5573232  0.539707 ]]
 Y: [[ 0.08422458 -0.86090374  0.37812304]
 [-0.00519627 -0.49453196  0.6178192 ]]

Model shape:
 (2, 3)

Weights:
 [[-0.40285552 -0.00293136  0.39895165]
 [-0.8884132  -0.78028464  0.23434246]
 [ 0.4721651  -0.60510516 -0.6874906 ]
 [-0.8282933   0.81940484 -0.49910602]]

Gradients:
 [[-0.6428592  -1.6074047   1.3670007 ]
 [-0.9208706  -0.77439123  0.58027476]
 [ 0.00811617 -0.35666528  0.3226366 ]
 [-0.7225579   0.26192445 -0.32874054]]


## Positional Encoding Layer


In [6]:
from keras.initializers import Constant


class PositionalEncoding(Layer):
    def __init__(self, max_seq_len, dim_model, **kwargs):
        super(PositionalEncoding, self).__init__(**kwargs)
        self.max_seq_len = max_seq_len
        self.dim_model = dim_model

    def build(self, input_shape):
        position = np.arange(self.max_seq_len)[:, np.newaxis]
        i = np.arange(self.dim_model)[np.newaxis, :]

        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(self.dim_model))
        angle_rads = position * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        positional_encoding = angle_rads[np.newaxis, ...]
        positional_encoding = tf.cast(positional_encoding, tf.float32)
        # self.positional_encoding = self.add_weight(
        #     name="positional_encoding",
        #     shape=(1, self.max_seq_len, self.dim_model),
        #     initializer=Constant(positional_encoding),
        #     trainable=False,
        #     dtype=tf.float32,
        # )
        self.positional_encoding = tf.Variable(
            positional_encoding, trainable=False, dtype=tf.float32
        )
        # self.positional_encoding.assign(positional_encoding)
        super(PositionalEncoding, self).build(input_shape)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        result = inputs + self.positional_encoding[:, :seq_len, :]
        return result

    def get_config(self):
        config = super(PositionalEncoding, self).get_config()
        config.update({"max_seq_len": self.max_seq_len, "dim_model": self.dim_model})
        return config

## Multi-Head Attention


In [7]:
class QuantizedMHA(Layer):
    def __init__(self, dim_model, head_num, quantize_to_bits=8, **kwargs):
        super(QuantizedMHA, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.head_num = head_num
        self.quantize_to_bits = quantize_to_bits

        assert dim_model % head_num == 0, "dim_model must be divisible by head_num"

        self.depth = dim_model // head_num
        self.q_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="q_layer"
        )
        self.k_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="k_layer"
        )
        self.v_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="v_layer"
        )
        self.out_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="output_layer"
        )

    def split_head(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.head_num, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None):
        batch_size = tf.shape(q)[0]

        q = self.q_dense(q)
        k = self.k_dense(k)
        v = self.v_dense(v)

        q = self.split_head(q, batch_size)
        k = self.split_head(k, batch_size)
        v = self.split_head(v, batch_size)

        qk = tf.matmul(q, k, transpose_b=True)
        dim_k = tf.cast(self.depth, tf.float32)
        scaled_attention_logits = qk / tf.sqrt(dim_k)

        if mask is not None:
            scaled_attention_logits += mask * -1e9

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        attention = tf.reshape(output, (batch_size, -1, self.dim_model))

        return self.out_dense(attention)

# Forwarding


In [8]:
class ForwardingNetwork(Layer):
    def __init__(self, dim_model, dim_forward, quantize_to_bits=8, **kwargs):
        super(ForwardingNetwork, self).__init__(**kwargs)
        self.dense_1 = QuantizedDense(
            units=dim_forward, quantize_to_bits=quantize_to_bits, name="forward_1"
        )
        self.dense_2 = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="forward_2"
        )

    def call(self, x):
        x = self.dense_1(x)
        x = tf.nn.gelu(x)
        return self.dense_2(x)

## Encoder-Decoder

- This one uses standard seq2seq architecture since it achieves higher accuracy for translation tasks


In [9]:
from keras.layers import LayerNormalization


class EncoderLayer(Layer):
    EPSILON = 1e-6

    def __init__(
        self,
        dim_model,
        dim_forward,
        head_num,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(EncoderLayer, self).__init__(**kwargs)
        self.mha = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.forwarding = ForwardingNetwork(dim_model, dim_forward, quantize_to_bits)
        self.layer_norm_1 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_2 = LayerNormalization(epsilon=self.EPSILON)

    def call(self, x, training=False, mask=None):
        attention = self.mha(x, x, x, mask)
        output_1 = self.layer_norm_1(x + attention)
        forwarding_output = self.forwarding(output_1)
        return self.layer_norm_2(output_1 + forwarding_output)


class DecoderLayer(Layer):
    EPSILON = 1e-6

    def __init__(
        self,
        dim_model,
        dim_forward,
        head_num,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(DecoderLayer, self).__init__(**kwargs)
        self.mha_1 = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.mha_2 = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.forwarding = ForwardingNetwork(dim_model, dim_forward, quantize_to_bits)
        self.layer_norm_1 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_2 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_3 = LayerNormalization(epsilon=self.EPSILON)

    def call(
        self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None
    ):
        attention_1 = self.mha_1(x, x, x, look_ahead_mask)
        output_1 = self.layer_norm_1(x + attention_1)

        attention_2 = self.mha_2(output_1, encoder_output, encoder_output, padding_mask)
        output_2 = self.layer_norm_2(output_1 + attention_2)

        forwarding_output = self.forwarding(output_2)
        return self.layer_norm_3(output_2 + forwarding_output)

In [10]:
def test_transformer():
    batch_size = 2
    seq_len = 10
    dim_model = 64
    head_num = 4
    dim_forward = 256

    test_x = tf.random.normal((batch_size, seq_len, dim_model))
    test_target = tf.random.normal((batch_size, seq_len, dim_model))
    encoder_layer = EncoderLayer(dim_model, dim_forward, head_num, quantize_to_bits=4)

    with tf.GradientTape() as tape:
        output = encoder_layer(test_x)
        loss = mean_squared_error(test_target, output)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, encoder_layer.trainable_variables)

    print("Encoder shape:\n", output.shape)
    print("\nTrainable weights len:\n", len(encoder_layer.trainable_variables))
    print("\nGradients:\n", gradients[0])


test_transformer()

Encoder shape:
 (2, 10, 64)

Trainable weights len:
 16

Gradients:
 tf.Tensor(
[[ 1.5084326e-05 -1.8236469e-04  9.5105241e-04 ...  2.8256129e-04
  -3.9297820e-04  1.0520481e-03]
 [-1.1419608e-03  9.9241745e-04  1.1402668e-03 ...  8.5978128e-05
   6.4438814e-04 -3.1716013e-04]
 [ 9.8247977e-04 -7.4364559e-04 -8.6312124e-04 ...  7.5836535e-05
  -1.0830747e-03 -9.4097405e-04]
 ...
 [-5.4140168e-04  7.6619594e-04  8.7510131e-04 ...  2.5792207e-04
   5.2954827e-04  7.0706895e-04]
 [-4.7694735e-04  1.7725823e-05  3.0002691e-04 ... -1.0720755e-03
   4.4371432e-04 -3.7527707e-04]
 [ 1.2542260e-03 -1.1563255e-03 -5.1076058e-04 ...  3.5782726e-04
  -1.5892095e-03  5.0757197e-04]], shape=(64, 64), dtype=float32)


## The Model Itself


In [11]:
def generate_padding_mask(seq):
    seq = tf.cast(tf.equal(seq, 0), tf.float32)
    return seq[:, tf.newaxis, tf.newaxis, :]


def generate_look_ahead_mask(seq_len):
    mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    return mask[tf.newaxis, tf.newaxis, :, :]

In [12]:
from keras.layers import Embedding


class Encoder(Layer):
    def __init__(
        self,
        layer_num,
        head_num,
        dim_model,
        dim_forward,
        vocab_size,
        max_seq_len,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(Encoder, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.layer_num = layer_num

        self.embedding = Embedding(vocab_size, dim_model)
        self.positional_encoding = PositionalEncoding(max_seq_len, dim_model)

        self.encoder_layers = [
            EncoderLayer(dim_model, dim_forward, head_num, quantize_to_bits)
            for _ in range(layer_num)
        ]

    def call(self, x, training=False, mask=None):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.sqrt(tf.cast(self.dim_model, tf.float32))
        x = self.positional_encoding(x)

        for i in range(self.layer_num):
            x = self.encoder_layers[i](x, training=training, mask=mask)
        return x


class Decoder(Layer):
    def __init__(
        self,
        layer_num,
        head_num,
        dim_model,
        dim_forward,
        vocab_size,
        max_seq_len,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(Decoder, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.layer_num = layer_num

        self.embedding = Embedding(vocab_size, dim_model)
        self.positional_encoding = PositionalEncoding(max_seq_len, dim_model)

        self.decoder_layers = [
            DecoderLayer(dim_model, dim_forward, head_num, quantize_to_bits)
            for _ in range(layer_num)
        ]

    def call(
        self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None
    ):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.sqrt(tf.cast(self.dim_model, tf.float32))
        x = self.positional_encoding(x)

        for i in range(self.layer_num):
            x = self.decoder_layers[i](
                x,
                encoder_output,
                training=training,
                look_ahead_mask=look_ahead_mask,
                padding_mask=padding_mask,
            )
        return x

In [13]:
from keras.models import Model


class PseudocodeTranslator(Model):
    def __init__(
        self,
        layer_num,
        head_num,
        dim_model,
        dim_forward,
        input_vocab_size,
        target_vocab_size,
        max_input_len,
        max_target_len,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(PseudocodeTranslator, self).__init__(**kwargs)
        self.encoder = Encoder(
            layer_num,
            head_num,
            dim_model,
            dim_forward,
            vocab_size=input_vocab_size,
            max_seq_len=max_input_len,
            quantize_to_bits=quantize_to_bits,
        )
        self.decoder = Decoder(
            layer_num,
            head_num,
            dim_model,
            dim_forward,
            vocab_size=target_vocab_size,
            max_seq_len=max_target_len,
            quantize_to_bits=quantize_to_bits,
        )
        self.final_layer = QuantizedDense(
            units=target_vocab_size,
            quantize_to_bits=quantize_to_bits,
            name="vocab_projection",
        )

    def generate_mask(self, input, target):
        encoder_padding_mask = generate_padding_mask(input)
        decoder_padding_mask = generate_padding_mask(input)
        decoder_target_padding_mask = generate_padding_mask(target)

        look_ahead_mask = generate_look_ahead_mask(tf.shape(target)[1])
        combined_mask = tf.maximum(decoder_target_padding_mask, look_ahead_mask)

        return encoder_padding_mask, decoder_padding_mask, combined_mask

    def call(self, io_pair, training=False):
        input, target = io_pair

        encoder_padding_mask, decoder_padding_mask, combined_mask = self.generate_mask(
            input, target
        )

        encoder_output = self.encoder(
            input, training=training, mask=encoder_padding_mask
        )
        decoder_output = self.decoder(
            target,
            encoder_output,
            training=training,
            look_ahead_mask=combined_mask,
            padding_mask=decoder_padding_mask,
        )

        final_output = self.final_layer(decoder_output)
        return final_output

In [14]:
def test_model():
    batch_size = 2
    input_seq_len = 20
    target_seq_len = 30

    model = PseudocodeTranslator(
        layer_num=2,
        head_num=4,
        dim_model=128,
        dim_forward=512,
        input_vocab_size=1000,
        target_vocab_size=2000,
        max_input_len=100,
        max_target_len=100,
        quantize_to_bits=4,
    )

    model.summary()

    test_input = tf.random.uniform(
        (batch_size, input_seq_len), minval=1, maxval=1000, dtype=tf.int32
    )
    test_target = tf.random.uniform(
        (batch_size, target_seq_len), minval=1, maxval=2000, dtype=tf.int32
    )

    logits = model((test_input, test_target))

    print("Test input shape:\n", test_input.shape)
    print("\nTest target shape:\n", test_target.shape)
    print("\nLogits shape:\n", logits.shape)
    # Expected shape (2,30,2000)


test_model()

Model: "pseudocode_translator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Encoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Decoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vocab_projection                │ ?                      │   0 (unbuilt) │
│ (QuantizedDense)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

c:\Users\shellbaby\miniconda3\envs\neural_network\Lib\site-packages\keras\src\layers\layer.py:1566: UserWarning: Layer 'pseudocode_translator' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: ''Exception encountered when calling Encoder.call().

Argument `initial_value` (Tensor("encoder_1/positional_encoding/Cast:0", shape=(1, 100, 128), dtype=float32)) could not be lifted out of a `tf.function`. (Tried to create variable with name='None'). To avoid this error, 

Test input shape:
 (2, 20)

Test target shape:
 (2, 30)

Logits shape:
 (2, 30, 2000)


# Synthetic AST

## Generator


In [25]:
from enum import IntEnum


class TYPE(IntEnum):
    assign = 1
    binary = 2
    print = 3


class ASTNode:
    pass


class VariableAssignment(ASTNode):
    def __init__(self, name, value):
        self.name = name
        self.value = value


class BinaryOperation(ASTNode):
    def __init__(self, target, left, operator, right):
        self.target = target
        self.left = left
        self.right = right
        self.operator = operator


class Print(ASTNode):
    def __init__(self, value):
        self.value = value


def generate_pseudocode(ast_nodes):
    lines = []
    for node in ast_nodes:
        if isinstance(node, VariableAssignment):
            lines.append(f"SET {node.name} TO {node.value}")
        elif isinstance(node, BinaryOperation):
            lines.append(
                f"CALCULATE {node.target} EQUALS {node.left} {node.operator} {node.right}"
            )
        elif isinstance(node, Print):
            lines.append(f"PRINT {node.value}")
    return "\n".join(lines)


def generate_csharp(ast_nodes):
    lines = []
    for node in ast_nodes:
        if isinstance(node, VariableAssignment):
            lines.append(f"int {node.name} = {node.value};")
        elif isinstance(node, BinaryOperation):
            lines.append(
                f"int {node.target} = {node.left} {node.operator} {node.right};"
            )
        elif isinstance(node, Print):
            lines.append(f"Console.WriteLine({node.value});")
    return "\n".join(lines)


def generate_synthetic_pseudocode(samples=1000):
    dataset: list[tuple] = []
    variables = ["x", "y", "i", "count", "total", "result", "a", "b", "c"]
    operators = ["+", "-", "*", "/"]

    for _ in range(samples):
        steps = random.randint(1, 3)
        nodes = []

        for _ in range(steps):
            type = random.choice(list(TYPE))
            if type.name == "assign":
                variable = random.choice(variables)
                value = random.randint(0, 100)
                nodes.append(VariableAssignment(variable, value))
            elif type.name == "binary":
                target = random.choice(variables)
                left = random.choice(variables)
                right = random.randint(0, 100)
                operator = random.choice(operators)
                nodes.append(BinaryOperation(target, left, operator, right))
            elif type.name == "print":
                value = random.choice(variables)
                nodes.append(Print(value))

        pseudocode = generate_pseudocode(nodes)
        real_code = generate_csharp(nodes)
        dataset.append((pseudocode, real_code))

    return dataset


def print_dataset(dataset: list[tuple], samples=10):
    for i, (pseudocode, real_code) in enumerate(dataset):
        if i >= samples:
            break
        print(f"\nPseudocode:\n", pseudocode)
        print(f"\nReal code:\n", real_code)


sample = generate_synthetic_pseudocode(samples=10)
print_dataset(dataset=sample)


Pseudocode:
 PRINT y
CALCULATE a EQUALS count / 91

Real code:
 Console.WriteLine(y);
int a = count / 91;

Pseudocode:
 CALCULATE count EQUALS i + 83
SET a TO 28

Real code:
 int count = i + 83;
int a = 28;

Pseudocode:
 PRINT c

Real code:
 Console.WriteLine(c);

Pseudocode:
 SET c TO 31
SET b TO 17

Real code:
 int c = 31;
int b = 17;

Pseudocode:
 PRINT c
PRINT result

Real code:
 Console.WriteLine(c);
Console.WriteLine(result);

Pseudocode:
 PRINT c
CALCULATE c EQUALS b / 20

Real code:
 Console.WriteLine(c);
int c = b / 20;

Pseudocode:
 CALCULATE count EQUALS total / 98
PRINT count

Real code:
 int count = total / 98;
Console.WriteLine(count);

Pseudocode:
 CALCULATE y EQUALS total * 30
CALCULATE result EQUALS c - 10

Real code:
 int y = total * 30;
int result = c - 10;

Pseudocode:
 SET a TO 88

Real code:
 int a = 88;

Pseudocode:
 PRINT count

Real code:
 Console.WriteLine(count);


## Tokenization


In [27]:
from keras.layers import TextVectorization


def standardize(input):
    standardized = tf.strings.lower(input)
    standardized = tf.strings.regex_replace(
        standardized, r"([;=+\-*/\(\)\{\}\[\]])", r" \1 "
    )
    standardized = tf.strings.regex_replace(standardized, r"\n", r" <NEWLINE> ")
    standardized = tf.strings.regex_replace(standardized, r"\s+", " ")
    return standardized


def tokenizer(dataset, vocab_size=1000, max_seq_len=100):
    pseudocode = [pair[0] for pair in dataset]
    real_code = [pair[1] for pair in dataset]

    pseudocode_vectorizer = TextVectorization(
        max_tokens=vocab_size,
        output_sequence_length=max_seq_len,
        standardize=standardize,
        split="whitespace",
    )

    real_code_vectorizer = TextVectorization(
        max_tokens=vocab_size,
        output_sequence_length=max_seq_len,
        standardize=standardize,
        split="whitespace",
    )

    pseudocode_vectorizer.adapt(pseudocode)
    real_code_vectorizer.adapt(real_code)

    return pseudocode_vectorizer, real_code_vectorizer

In [31]:
def test_standardization():
    test_code = "int x = 45;\nConsole.Writeline(x);"
    standardized = standardize(test_code)
    print(f"Original:\n{test_code}")
    print(f"\nStandardized: {standardized}")
test_standardization()

Original:
int x = 45;
Console.Writeline(x);

Standardized: b'int x = 45 ; <NEWLINE> console.writeline ( x ) ; '
